In [ ]:
# Import packages
import os
from matplotlib import pyplot as plt
import pandas as pd
import datetime
import matplotlib as mpl

# Import AuTuMN modules
from autumn.settings import Models, Region
from autumn.settings.folders import OUTPUT_DATA_PATH
from autumn.core.project import get_project
from autumn.core import db
from autumn.core.plots.calibration.plots import calculate_r_hats, get_output_from_run_id
from autumn.core.plots.uncertainty.plots import _plot_uncertainty, _get_target_values
from autumn.core.plots.plotter.base_plotter import COLOR_THEME
from autumn.core.plots.utils import get_plot_text_dict, change_xaxis_to_date, REF_DATE, ALPHAS, COLORS, _apply_transparency, _plot_targets_to_axis

from autumn.calibration.utils import get_uncertainty_df

In [ ]:
# Specify model details
model = Models.SM_SIR
region = Region.MALAYSIA
dirname = "2023-02-14"

In [ ]:
p = get_project("sm_sir", "malaysia")

In [ ]:
m = p.build_model(p.param_set.baseline.to_dict())

In [ ]:
for s in m._stratifications:
    print(s)
    print(s.strata)
    #print(s.flow_adjustments)

In [ ]:
m._stratifications

In [ ]:
from autumn.core.runs import ManagedRun

In [ ]:
run_id = "sm_sir/malaysia/03072025/2hourfrom14mle"

In [ ]:
mr = ManagedRun(run_id)

In [ ]:
db = mr.powerbi.get_db()

In [ ]:
db.scenarios

In [ ]:
do = db.get_derived_outputs()

In [ ]:
from matplotlib import pyplot as plt

In [ ]:

figsize = (14.0,35/3.)

In [ ]:
scenario_title_strings = [
    "Baseline",
    "No vaccination",
    "Half vaccination coverage",
    "No nationwide MCO",
    "No recovery MCO",
    "No nationwide/recovery MCO",
    "No vaccination and no MCOs"
]

scenario_legend_strings = [
    "Baseline",
    "No vaccination",
    "50% vaccination coverage",
    "Nationwide MCO not implemented",
    "Recovery MCO not implemented",
    "Nationwide and recovery MCO not implemented",
    "No vaccination and no MCOs"
]

In [ ]:
row_scenarios = [3,4,5]

In [ ]:
col_datasets = [
    do["notifications"].loc[:"30 sep 2022"],
    do["hospital_occupancy"].loc[:"30 sep 2022"],
    do["infection_deaths"].cumsum().loc[:"30 sep 2022"]
]

In [ ]:
col_ylims = [
    (0.0,70000.0),
    (0.0,40000.0),
    (0.0,100000.0),
]

In [ ]:
figsize = (14*1.5,52.5/3)
f0, f1 = figsize
figsizes = {
    "MCO": figsize,
    "vacc_MCO": figsize,
    "vacc": (f0, f1*(2/3))
}

In [ ]:
from matplotlib import ticker

In [ ]:
plot_specs = {
    "MCO": {
        "colors": [
            ["blue", "gold"],
            ["blue", "black"],
            ["blue", "orangered"]
        ],
        "row_scenarios": [3,4,5]
    },
    "vacc_MCO": {
        "colors": [
            ["blue", "magenta"],
            ["blue", "orangered"],
            ["blue", "darkgreen"]
        ],
        "row_scenarios": [1,5,6]
    },
    "vacc": {
        "colors": [
            ["blue", "magenta"],
            ["blue", "palegreen"]
        ],
        "row_scenarios": [1, 2]
    }
}

In [ ]:
col_labels = [
    "Daily number of cases",
    "Hospital beds occupied",
    "Cumulative number of deaths"
]

In [ ]:
import matplotlib.dates as mdates

In [ ]:
plot_key = "MCO"
pspec = plot_specs[plot_key]
row_scenarios = pspec["row_scenarios"]
pcolors = pspec["colors"]
figsize = figsizes[plot_key]

fig = plt.figure(figsize=figsize, dpi=600, layout="constrained")
#fig.suptitle('Figure title')
plt.style.use('seaborn-v0_8-whitegrid')

# create 3x1 subfigs

nrows = 3 if plot_key != "vacc" else 2

subfigs = fig.subfigures(nrows=nrows, ncols=1)
for row, subfig in enumerate(subfigs):
    #subfig.suptitle(f'Subfigure title {row}', ha="left")
    scenario = row_scenarios[row]

    row_str = ["A","B","C"][row]

    colors = pcolors[row]

    # create 1x3 subplots per subfig
    axs = subfig.subplots(nrows=1, ncols=3)
    for col, ax in enumerate(axs):
        #ax.plot()

        lw = 4
        
        if col == 0:
            ax.set_title(f"{row_str}. {scenario_title_strings[scenario]}", loc="left", fontsize=20, pad=20)
            legends = ["Baseline", scenario_legend_strings[scenario]]
            col_datasets[col][[0,scenario]].plot(ax=ax,ylim=col_ylims[col], lw=lw,color=colors)
            ax.legend(legends, loc="upper left")
        else:
            col_datasets[col][[0,scenario]].plot(ax=ax,ylim=col_ylims[col],legend=None, lw=lw,color=colors)
        ax.yaxis.set_major_formatter(ticker.EngFormatter())
        ax.set_ylabel(col_labels[col], fontsize=15)
        ax.tick_params(axis='both', which='major', labelsize=15)
        ax.tick_params(axis='both', which='minor', labelsize=15)
        ax.tick_params(axis='x', labelrotation=90)
        ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=range(1,13,2)))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
        ax.xaxis.set_minor_locator(mdates.MonthLocator())
fig.savefig("MCO_scenario_plots_update88.pdf")#,bbox_inches='tight')
fig.savefig("MCO_scenario_plots_update88.png")

In [ ]:
targets = db.get_targets()

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
from autumn.core.project import load_timeseries

In [ ]:
from autumn.settings.constants import COVID_BASE_DATETIME

In [ ]:
from summer.utils import ref_times_to_dti

In [ ]:
ts = load_timeseries("../../../autumn/projects/sm_sir/malaysia/malaysia/timeseries.json")

In [ ]:
tkeys = [
    "notifications",
    "hospital_occupancy",
    "icu_occupancy",
    "infection_deaths"
]


In [ ]:
ts["notifications"].to_numpy()

In [ ]:
target_series = {}

for k in tkeys:
    traw = ts[k]
    newidx = ref_times_to_dti(COVID_BASE_DATETIME, traw.index)
    target_series[k] = pd.Series(data=traw.to_numpy(), index=newidx).loc["1 jan 2021": "30 sep 2022"]


In [ ]:
ref_times_to_dti(COVID_BASE_DATETIME, ts["notifications"].index)

In [ ]:
figsize = figsizes["vacc"]

fig = plt.figure(figsize=figsize, dpi=600, layout="constrained")
#fig.suptitle('Figure title')
plt.style.use('seaborn-v0_8-whitegrid')

# create 3x1 subfigs
colors = {
    "inner": "#4646bab7",
    "outer": "#5a69be5f"
}

nrows = 2
ax_labels = [
    "Daily number of cases",
    "Hospital beds occupied",
    "ICU beds occupied",
    "Daily number of deaths"
]

ylims = [ 
    (0,50000),
    (0,20000),

]

axs = fig.subplots(nrows=2, ncols=2)
for i, ax in enumerate(axs.flat):
    #ax.plot()

    lw = 4

    k = tkeys[i]

    qdf = db.get_uncertainty()[k][0].loc[:"30 sep 2022"]
    qdf[[0.025,0.975]].plot(color=colors["outer"],legend=None, ax=ax)
    ax.fill_between(qdf.index, qdf[0.025], qdf[0.975],color=colors["outer"])
    qdf[[0.25,0.75]].plot(color=colors["inner"],ax=ax,legend=None)
    ax.fill_between(qdf.index, qdf[0.25], qdf[0.75],color=colors["inner"])
    qdf[0.5].plot(color="#2222cc99",legend=False,ax=ax)
    target_series[k].loc[:"30 sep 2022"].plot(style='.',color="#00000088",ax=ax)

    ax.yaxis.set_major_formatter(ticker.EngFormatter())
    ax.set_ylabel(ax_labels[i], fontsize=15)
    ax.tick_params(axis='both', which='major', labelsize=15)
    ax.tick_params(axis='both', which='minor', labelsize=15)
    ax.tick_params(axis='x', labelrotation=90)
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=range(1,13,2)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_minor_locator(mdates.MonthLocator())

fig.savefig("baseline_calibration_plots_update88.pdf")#,bbox_inches='tight')
fig.savefig("baseline_calibration_plots_update88.png")

In [ ]:
db.get_

In [ ]:
k = "hospital_occupancy"

qdf = db.get_uncertainty()[k][0]
plt = qdf[[0.025,0.975]].plot(color=colors["outer"],legend=None)
plt.fill_between(qdf.index, qdf[0.025], qdf[0.975],color=colors["outer"])
plt = qdf[[0.25,0.75]].plot(color=colors["inner"],ax=plt,legend=None)
plt.fill_between(qdf.index, qdf[0.25], qdf[0.75],color=colors["inner"])
qdf[0.5].plot(color="#2222cc99",legend=False,ax=plt)
targets[k].plot(style='.',color="#00000088")

In [ ]:
from os import listdir as ls

In [ ]:
ls(OUTPUT_DATA_PATH / "runs" / "sm_sir/malaysia/02072025/14hourfrommle/data/calibration_outputs")

In [ ]:
# get the relevant project and output data
project = get_project(model, region)
#project_calib_dir = os.path.join(
#    OUTPUT_DATA_PATH, "calibrate", project.model_name, project.region_name
#)
#calib_path = os.path.join(project_calib_dir, dirname)
calib_path = OUTPUT_DATA_PATH / "runs" / "sm_sir/malaysia/02072025/14hourfrommle/data/calibration_outputs"
# Load tables
mcmc_tables = db.load.load_mcmc_tables(calib_path)
mcmc_params = db.load.load_mcmc_params_tables(calib_path)

uncertainty_df = get_uncertainty_df(calib_path, mcmc_tables, project.plots)
scenario_list = uncertainty_df['scenario'].unique()

# make output directories
output_dir = f"mea_plots"
base_dir = os.path.join("outputs", output_dir)
os.makedirs(base_dir, exist_ok=True)
dirs_to_make = ["calibration", "MLE", "median", "uncertainty", "csv_files"]
for dir_to_make in dirs_to_make:
    os.makedirs(os.path.join(base_dir, dir_to_make), exist_ok=True)

In [ ]:
# get R_hat diagnostics
# r_hats = calculate_r_hats(mcmc_params, mcmc_tables, burn_in=0)
# for key, value in r_hats.items():
#     print(f"{key}: {value}")

In [ ]:
uncertainty_df.columns

In [ ]:
titles = {
    "notifications": "Daily number of notified Covid-19 cases",
    "infection_deaths": "Daily number of Covid-19 deaths",
     #"cumulative_infection_deaths": "Cumulative number of Covid-19 deaths",
    #"incidence": "Daily incidence (incl. asymptomatics and undetected)",
    "hospital_occupancy": "Hospital beds occupied by Covid-19 patients",
    "icu_occupancy": "ICU beds occupied by Covid-19 patients",
    #"cdr": "Proportion detected among symptomatics",
    #"proportion_vaccinated": "Proportion vaccinated",
    #"prop_incidence_strain_delta": "Proportion of Delta variant in new cases",
    ##"prop_incidence_strain_alpha_beta":  "Proportion of Alpha variant in new cases",
    #"prop_ever_infected": "Proportion ever infected",
    #"prop_detected_traced": "Proportion of cases contact traced"
}
def plot_outputs(output_type, output_name, scenario_list, sc_linestyles, sc_colors, show_v_lines=False, x_min=590, x_max=775):

    title = titles[output_name]
    title_fontsize = 24
    label_font_size = 24
    linewidth = 3
    n_xticks = 10
    legend = True

    # initialise figure
    fig = plt.figure(figsize=(12, 8))
    plt.style.use("ggplot")
    axis = fig.add_subplot()

    # prepare colors for ucnertainty
    n_scenarios_to_plot = len(scenario_list)
    uncertainty_colors = _apply_transparency(COLORS[:n_scenarios_to_plot], ALPHAS[:n_scenarios_to_plot])

    
    if output_type == "MLE":
        derived_output_tables = db.load.load_derived_output_tables(calib_path, column=output_name)
        
    for i, scenario in enumerate(scenario_list):    
        linestyle = sc_linestyles[scenario]
        color = sc_colors[scenario]

        if output_type == "MLE":
            times, values = get_output_from_run_id(output_name, mcmc_tables, derived_output_tables, "MLE", scenario)
            axis.plot(times, values, color=color, linestyle=linestyle, linewidth=linewidth)
            quantiles = 0
        elif output_type == "median":
            _plot_uncertainty(
                axis,
                uncertainty_df,
                output_name,
                scenario,
                x_max,
                x_min,
                [_, _, _, color],
                overlay_uncertainty=True,
                start_quantile=0,
                zorder=scenario + 1,
                linestyle=linestyle,
                linewidth=linewidth,
             )
        elif output_type == "uncertainty":
            scenario_colors = uncertainty_colors[i]  

            times, quantiles = _plot_uncertainty(
                axis,
                uncertainty_df,
                output_name,
                scenario,
                x_max,
                x_min,
                scenario_colors,
                overlay_uncertainty=True,
                start_quantile=0,
                zorder=scenario + 1,
                
             )
                  

        else:
            print("Please use supported output_type option")

        if output_name == "notifications":
            if legend:
                ax = plt.gca()
                legend_elem = [mpl.patches.Patch(facecolor=uncertainty_colors[0][1], label='baseline'),
#                                mpl.patches.Patch(facecolor=uncertainty_colors[1][3], 
#                                                     label='No vaccination'),
#                                mpl.patches.Patch(facecolor=uncertainty_colors[2][3], 
#                                                     label='Vaccine coverage halved'),
#                                mpl.patches.Patch(facecolor=uncertainty_colors[1][1], 
#                                                     label='Nation-wide MCO not implemented'),
#                                mpl.patches.Patch(facecolor=uncertainty_colors[2][2], 
#                                                     label='Recovery MCO not implemented'),
#                                mpl.patches.Patch(facecolor=uncertainty_colors[1][2], 
#                                                    label='Recovery and nation-wide MCOs not implemented'),
#                                  mpl.patches.Patch(facecolor=uncertainty_colors[3][2], 
#                                                     label='No vaccination and MCOs not implemented'),
                              ]
                ax.legend(handles=legend_elem, fontsize = 16, loc = "upper left")

    axis.set_xlim((x_min, x_max))
    axis.set_title(title, fontsize=title_fontsize)
    plt.setp(axis.get_yticklabels(), fontsize=label_font_size)
    plt.setp(axis.get_xticklabels(), fontsize=label_font_size)
    change_xaxis_to_date(axis, REF_DATE)
    plt.locator_params(axis="x", nbins=n_xticks)

          
    return axis, quantiles, times

# Scenario plots with single lines

In [ ]:
output_names = ["notifications","icu_occupancy","hospital_occupancy", "infection_deaths"]#"cumulative_infection_deaths"]

scenario_x_min, scenario_x_max = 426, 975
scenarios_to_plot = [0]
sc_colors = [COLOR_THEME[i] for i in scenario_list]
sc_linestyles = ["dotted"] + ["solid"] * (len(scenario_list)-1)
for output_type in ["MLE"]:
    for output_name in output_names:
        plot_outputs(output_type, output_name, scenarios_to_plot, sc_linestyles, sc_colors, False, x_min=scenario_x_min, x_max=scenario_x_max)
        

#          path = os.path.join(base_dir, output_type, f"{output_name}.png")
#         plt.savefig(path)

# Uncertainty around scenarios


In [ ]:
def to_date(x_value, date_str_format="%#d-%b-%Y"):
    ref_date = datetime.date(2019, 12, 31)
    date = ref_date + datetime.timedelta(days=int(x_value))
    return date.strftime(date_str_format)

In [ ]:
db = mr.powerbi.get_db()

In [ ]:
db.get_derived_outputs().columns.get_level_values(0).unique()

In [ ]:
from datetime import  timedelta

In [ ]:
def get_dti_equiv(s):
    out_idx = pd.DatetimeIndex([REF_DATE + timedelta(t) for t in s.index])
    s_out = s.copy()
    s_out.index = out_idx
    return s_out

In [ ]:
get_dti_equiv(project.calibration.targets[2].data)

In [ ]:
tmap = {i:t.data.name for i,t in enumerate(project.calibration.targets)}#[3].data.name

In [ ]:
target = 3

db.get_derived_outputs()[tmap[target]][0].plot()
get_dti_equiv(project.calibration.targets[target].data).plot()

In [ ]:
db.get_uncertainty()["hospital_occupancy"][0].plot()
get_dti_equiv(project.calibration.targets[2].data).plot()

In [ ]:
output_type = "uncertainty"
for output_name in output_names:
    axis, quantiles, times = plot_outputs(output_type, output_name, scenarios_to_plot, sc_linestyles, sc_colors, False, x_min=scenario_x_min, x_max=scenario_x_max)
    quantile_val = 0.025
  
    quantile_max_val= (quantiles[quantile_val])## to get the maximum of median and 95% range 0.5, 0.025,0.975
    quantile_max_val = max(quantile_max_val)#[-1]
    index_quantile= quantiles[quantile_val].index(quantile_max_val)
    time_quantile = to_date(times[index_quantile])
    print(output_name+f" maximum quantile value:", quantile_max_val) 
    print(output_name+f" time corresponding to maximum:",time_quantile )
    targets = project.plots
    targets = {k: v for k, v in targets.items() if v["output_key"] == output_name}
    values, times = _get_target_values(targets, output_name)
    axis.scatter(times, values, marker="o", color="black", s=10, zorder=999)
    _plot_targets_to_axis(axis, values, times, on_uncertainty_plot=True)
    
    #path = os.path.join(base_dir, output_type, f"{output_name}_scenario_{scenario}.png")
    #plt.savefig(path)

# Uncertainty around baseline only (calibration plots)

In [ ]:
calibration_x_min, calibration_x_max = 640, 1003

for output_name in output_names + ["cdr","prop_ever_infected"]:
    axis, quantiles, t = plot_outputs("uncertainty", output_name, [0], sc_linestyles, sc_colors, False,  x_min=calibration_x_min, x_max=calibration_x_max)  
    #path = os.path.join(base_dir, 'calibration', f"{output_name}.png")
 
#     targets = project.plots
#     targets = {k: v for k, v in targets.items() if v["output_key"] == output_name}
#     values, times = _get_target_values(targets, output_name)
#     axis.scatter(times, values, marker="o", color="black", s=10, zorder=999)
#     _plot_targets_to_axis(axis, values, times, on_uncertainty_plot=True)
    
    #plt.savefig(path)


# outputs to csv files

In [ ]:
# csv_outputs = ["icu_occupancy"]
# start_time = 609 # 31 Aug 2021

# includes_MLE = True
# requested_quantiles = [0.025, 0.50, 0.975]

# # for age in [str(int(5. * i)) for i in range(16)]:
# #     csv_outputs.append(f"notificationsXagegroup_{age}")

# def get_uncertainty_data(output_name, scenario_idx, quantile):
#     mask = (
#             (uncertainty_df["type"] == output_name)
#             & (uncertainty_df["scenario"] == scenario_idx)
#             & (uncertainty_df["quantile"] == quantile)
#         )
#     df = uncertainty_df[mask]
#     times = df.time.unique()[1:]
#     values = df["value"].tolist()[1:]
        
#     return times, values

# COVID_BASE_DATE = pd.datetime(2019, 12, 31)
# start_date = pd.to_timedelta(start_time, unit="days") + (COVID_BASE_DATE)  

# for scenario in scenario_list:
#     df = pd.DataFrame()
    
#     # include a column for the date
#     t, _ = get_uncertainty_data("notifications", scenario, 0.5)
#     df["date"] = pd.to_timedelta(t, unit="days") + (COVID_BASE_DATE)  
    
#     for output in csv_outputs:
#         if includes_MLE:
#             derived_output_tables = db.load.load_derived_output_tables(calib_path, column=output)
#             do_times, do_values = get_output_from_run_id(output, mcmc_tables, derived_output_tables, "MLE", scenario)            
            
#             assert list(do_times[1:]) == list(t)
#             do_values = list(do_values)[1:]        

#             name = f"{output}_MLE"
#             df[name] = do_values       
       
#         if output in list(uncertainty_df["type"].unique()):
#             for quantile in requested_quantiles:
#                 _, v = get_uncertainty_data(output, scenario, quantile)         
#                 name = f"{output}_{quantile}"
#                 df[name] = v            
    
    
#     # trim the dataframe to keep requested times only
#     df.drop(df[df.date < start_date].index, inplace=True)    
    
#     path = os.path.join(base_dir, 'csv_files', f"outputs_scenario_{scenario}.csv")
#     df.to_csv(path)
            


In [ ]:
from summer.utils import ref_times_to_dti
ref_times_to_dti(REF_DATE, [1003])